# flashattention-cuda — Colab bootstrap (T4)

One pass: confirm the GPU → build the kernel → predict the roofline → test vs SDPA → benchmark.

**Runtime → Change runtime type → T4 GPU** before running. Everything below runs on the GPU;
nothing here works on a CPU-only runtime.

## 0. Confirm the hardware (Step 0 of the brief)
We record GPU model, compute capability, and clocks — every benchmark row must carry these,
and the free-tier T4 thermally throttles.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,clocks.sm,clocks.max.sm,memory.total,temperature.gpu --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| capability', torch.cuda.get_device_capability())

## 1. Get the repo
`REPO_URL` is already set to the public repo, so the clone just works. (If you fork it, point
`REPO_URL` at your fork.) The repo root is added to `sys.path` so the notebook can import it.

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works on Colab
import os, sys, subprocess
if not os.path.isdir('flashattention-cuda'):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
os.chdir('flashattention-cuda')
sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Predict the roofline BEFORE running anything
The per-step loop starts here: predict the limiter, then check it against reality below.

In [ ]:
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x64 --precision fp32 --materialize-s
print()
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x128 --precision fp32 --materialize-s

## 3. Build the v1 kernel (JIT)
First call compiles with nvcc (~1 min); cached afterwards. `verbose=True` prints the build.

In [ ]:
# Clean build: a failed compile (e.g. ninja missing on the first try) leaves a stale cache dir
# with a version stamp but no .so, so torch SKIPS the rebuild and then fails to import. Remove
# any partial fa_* build dir (one with no compiled .so); a good cached build is kept, so re-runs
# stay fast.
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_*')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True)
        print('cleaned stale build:', d)
print('clean-build check done')

In [ ]:
# cpp_extension.load() compiles via ninja, which Colab doesn't always ship — install it first.
!pip install -q ninja
from bindings.load import build_kernel
mod = build_kernel('v1_naive')
print('built:', mod)

## 4. Correctness vs SDPA (documented tolerance: atol/rtol 1e-4)

In [ ]:
!python -m pytest tests/ -q

## 5. Benchmark vs SDPA across the sweep
Expect to be **slower than SDPA** — SDPA is already a fused efficient kernel. This is the
'before'. Paste the numbers into `docs/results.md` and compare to the roofline prediction.

In [ ]:
!python -m bench.harness --backend v1_naive --precision fp32

## 6. (Optional) Nsight Compute capture
Confirms the limiter: DRAM throughput near peak at d=64 (the bandwidth wall), low MMA/MUFU.
`ncu` may need a GPU runtime that allows profiling; see profiling/GUIDE.md.

In [ ]:
!bash profiling/capture.sh v1_naive || echo 'ncu unavailable on this runtime; read GUIDE.md'